# Chatbot Conversation Logs Analysis

This notebook analyzes chatbot conversation logs to understand:
1. Time range of the data
2. Message frequency and quantity patterns per user
3. Conversation groupings
4. Query classification (well-specified vs. mispecified)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from openai import OpenAI
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
# Load the conversation logs
df = pd.read_csv('chatbot_conversation_logs.csv')
print(f"Total records: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## 1. Time Range Analysis

In [ ]:
# Convert message_timestamp to datetime
df['message_timestamp'] = pd.to_datetime(df['message_timestamp'], format='%m/%d/%Y %H:%M:%S')
df['message_date'] = pd.to_datetime(df['message_date'], format='%m/%d/%Y')

# Time range analysis
min_date = df['message_timestamp'].min()
max_date = df['message_timestamp'].max()
date_range = max_date - min_date

print("="*60)
print("TIME RANGE ANALYSIS")
print("="*60)
print(f"Earliest message: {min_date}")
print(f"Latest message: {max_date}")
print(f"Date range: {date_range.days} days ({date_range.days / 30:.1f} months)")
print(f"\nTotal messages: {len(df):,}")
print(f"Messages per day (avg): {len(df) / max(date_range.days, 1):.1f}")

In [ ]:
# Messages over time
daily_messages = df.groupby(df['message_date'].dt.date).size()

plt.figure(figsize=(14, 5))
plt.plot(daily_messages.index, daily_messages.values, linewidth=0.8)
plt.title('Daily Message Volume Over Time')
plt.xlabel('Date')
plt.ylabel('Number of Messages')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nDaily message statistics:")
print(f"  Min: {daily_messages.min()}")
print(f"  Max: {daily_messages.max()}")
print(f"  Mean: {daily_messages.mean():.1f}")
print(f"  Median: {daily_messages.median():.1f}")

## 2. Per-User Message Frequency and Quantity Analysis

In [ ]:
# Basic user statistics
unique_users = df['user_id'].nunique()
print("="*60)
print("USER OVERVIEW")
print("="*60)
print(f"Total unique users: {unique_users:,}")

In [ ]:
# Per-user message analysis
user_stats = df.groupby('user_id').agg({
    'message': 'count',
    'message_timestamp': ['min', 'max'],
    'sender': lambda x: (x == 'USER').sum()  # Count user messages only
}).reset_index()

user_stats.columns = ['user_id', 'total_messages', 'first_message', 'last_message', 'user_messages']
user_stats['bot_messages'] = user_stats['total_messages'] - user_stats['user_messages']
user_stats['active_duration_days'] = (user_stats['last_message'] - user_stats['first_message']).dt.days
user_stats['messages_per_day'] = user_stats['total_messages'] / user_stats['active_duration_days'].replace(0, 1)

print("\nPer-user message statistics:")
print(f"  Total messages - Mean: {user_stats['total_messages'].mean():.1f}, Median: {user_stats['total_messages'].median():.1f}")
print(f"  User messages - Mean: {user_stats['user_messages'].mean():.1f}, Median: {user_stats['user_messages'].median():.1f}")
print(f"  Bot messages - Mean: {user_stats['bot_messages'].mean():.1f}, Median: {user_stats['bot_messages'].median():.1f}")
print(f"  Active duration (days) - Mean: {user_stats['active_duration_days'].mean():.1f}, Median: {user_stats['active_duration_days'].median():.1f}")

In [ ]:
# Distribution of messages per user
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total messages distribution
axes[0].hist(user_stats['total_messages'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Total Messages per User')
axes[0].set_xlabel('Number of Messages')
axes[0].set_ylabel('Number of Users')

# User messages distribution
axes[1].hist(user_stats['user_messages'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('Distribution of User Messages per User')
axes[1].set_xlabel('Number of User Messages')
axes[1].set_ylabel('Number of Users')

# Active duration distribution
axes[2].hist(user_stats['active_duration_days'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[2].set_title('Distribution of Active Duration')
axes[2].set_xlabel('Days Active')
axes[2].set_ylabel('Number of Users')

plt.tight_layout()
plt.show()

In [ ]:
# Top users by message count
print("\nTop 10 users by total messages:")
top_users = user_stats.nlargest(10, 'total_messages')[['user_id', 'total_messages', 'user_messages', 'bot_messages', 'active_duration_days']]
print(top_users.to_string(index=False))

In [ ]:
# User engagement buckets
def categorize_engagement(msg_count):
    if msg_count <= 5:
        return '1-5 messages (Low)'
    elif msg_count <= 20:
        return '6-20 messages (Medium)'
    elif msg_count <= 50:
        return '21-50 messages (High)'
    else:
        return '50+ messages (Very High)'

user_stats['engagement_level'] = user_stats['total_messages'].apply(categorize_engagement)
engagement_dist = user_stats['engagement_level'].value_counts().sort_index()

print("\nUser Engagement Distribution:")
for level, count in engagement_dist.items():
    pct = count / len(user_stats) * 100
    print(f"  {level}: {count:,} users ({pct:.1f}%)")

## 3. Conversation Grouping and Display

In [ ]:
# Group messages into conversations
# A conversation is defined as messages from a user with gaps of less than 30 minutes

def group_into_conversations(user_df, gap_threshold_minutes=30):
    """Group messages into conversations based on time gaps."""
    user_df = user_df.sort_values('message_timestamp')
    
    conversations = []
    current_conversation = []
    last_timestamp = None
    
    for _, row in user_df.iterrows():
        if last_timestamp is None:
            current_conversation.append(row)
        elif (row['message_timestamp'] - last_timestamp).total_seconds() / 60 > gap_threshold_minutes:
            # New conversation
            if current_conversation:
                conversations.append(current_conversation)
            current_conversation = [row]
        else:
            current_conversation.append(row)
        
        last_timestamp = row['message_timestamp']
    
    if current_conversation:
        conversations.append(current_conversation)
    
    return conversations

# Group all conversations
all_conversations = []
for user_id in tqdm(df['user_id'].unique(), desc="Grouping conversations"):
    user_df = df[df['user_id'] == user_id]
    user_conversations = group_into_conversations(user_df)
    for conv in user_conversations:
        all_conversations.append({
            'user_id': user_id,
            'messages': conv,
            'num_messages': len(conv),
            'start_time': conv[0]['message_timestamp'],
            'end_time': conv[-1]['message_timestamp']
        })

print(f"\nTotal conversations identified: {len(all_conversations):,}")
print(f"Average conversations per user: {len(all_conversations) / unique_users:.1f}")

In [ ]:
# Conversation statistics
conv_lengths = [c['num_messages'] for c in all_conversations]
print("\nConversation length statistics:")
print(f"  Min: {min(conv_lengths)}")
print(f"  Max: {max(conv_lengths)}")
print(f"  Mean: {np.mean(conv_lengths):.1f}")
print(f"  Median: {np.median(conv_lengths):.1f}")

# Distribution of conversation lengths
plt.figure(figsize=(10, 4))
plt.hist(conv_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.title('Distribution of Conversation Lengths')
plt.xlabel('Number of Messages per Conversation')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Initialize OpenAI client for translations
client = OpenAI()

def translate_to_english(client, text, model_name="gpt-4.1-mini"):
    """
    Translate text to English using GPT. Returns original if already in English.
    """
    prompt = f"""Translate the following text to English. If the text is already in English, return it as-is.
Only return the translated text, nothing else.

Text: "{text}"

Translation:"""

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=500
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error translating: {e}")
        return text

In [ ]:
# Display sample conversations (with translation for ALL messages)
def print_conversation(conv, client, max_messages=20):
    """Pretty print a conversation with translations for all messages."""
    print(f"\n{'='*70}")
    print(f"User ID: {conv['user_id']}")
    print(f"Time: {conv['start_time']} to {conv['end_time']}")
    print(f"Messages: {conv['num_messages']}")
    print("-"*70)
    
    for i, msg in enumerate(conv['messages'][:max_messages]):
        sender = msg['sender']
        message = str(msg['message'])
        timestamp = msg['message_timestamp']
        
        prefix = "[USER]" if sender == 'USER' else "[BOT] "
        
        # Translate the message to English
        if len(message) > 2:
            translated = translate_to_english(client, message, model_name="gpt-4.1-mini")
            # Truncate for display
            translated_display = translated[:300] + "..." if len(translated) > 300 else translated
            print(f"{prefix} ({timestamp.strftime('%H:%M:%S')}): {translated_display}")
        else:
            print(f"{prefix} ({timestamp.strftime('%H:%M:%S')}): {message}")
    
    if conv['num_messages'] > max_messages:
        print(f"\n... ({conv['num_messages'] - max_messages} more messages)")

# Show some sample conversations with substantive content
print("\nSAMPLE CONVERSATIONS (translated to English)")
print("="*70)

# Filter for conversations with at least 5 messages to show meaningful exchanges
substantive_convs = [c for c in all_conversations if c['num_messages'] >= 5]

# Show 5 random sample conversations
import random
random.seed(42)
sample_convs = random.sample(substantive_convs, min(5, len(substantive_convs)))

for conv in sample_convs:
    print_conversation(conv, client)

## 4. Query Classification: Well-Specified vs. Mispecified

Using GPT 5 mini to classify user queries as well-specified or vague/mispecified.

In [ ]:
# Overview of user messages
user_messages_df = df[df['sender'] == 'USER']
print(f"Total user messages: {len(user_messages_df):,}")

In [ ]:
# Classification function (client and translate_to_english already defined above)

def classify_query_clarity(client, question, model_name="gpt-4.1-mini"):
    """
    Use GPT model to classify if a question is well-specified or vague/mispecified.
    The question should already be translated to English.
    """
    prompt = f"""You are a medical health chatbot answering user's questions about pregnancy. 
Analyze the following question and determine if it's well-specified or too vague/mispecified to answer properly.

Question: "{question}"

Instructions:
- WELL_SPECIFIED: The question is clear, specific, and can be answered accurately. It contains enough context and details.
- MISPECIFIED: The question is too vague, lacks context, is ambiguous, or needs clarification to provide an accurate answer.

Respond with ONLY one word: either "WELL_SPECIFIED" or "MISPECIFIED"

Response:"""

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=10
        )
        
        classification = response.choices[0].message.content.strip().upper()
        
        if "WELL_SPECIFIED" in classification:
            return "WELL_SPECIFIED"
        elif "MISPECIFIED" in classification or "VAGUE" in classification:
            return "MISPECIFIED"
        else:
            return "UNCLEAR"
            
    except Exception as e:
        print(f"Error classifying question: {e}")
        return "ERROR"

In [ ]:
# Classify a sample of questions using GPT 5 mini
# Also capture the bot response for each user query

# First, create a mapping of user messages to their subsequent bot responses
def get_bot_response_for_user_message(df, user_msg_idx):
    """Find the bot response that follows a user message."""
    # Look for the next BOT message after this user message
    for i in range(user_msg_idx + 1, min(user_msg_idx + 5, len(df))):
        if df.iloc[i]['sender'] == 'BOT':
            return df.iloc[i]['message']
    return None

# Create a dataframe with user messages and their indices
df_sorted = df.sort_values(['user_id', 'message_timestamp']).reset_index(drop=True)
user_msg_indices = df_sorted[df_sorted['sender'] == 'USER'].index.tolist()

# Build list of user questions with their bot responses
questions_with_responses = []
for idx in user_msg_indices:
    user_msg = df_sorted.iloc[idx]['message']
    if not user_msg or not str(user_msg).strip():
        continue
    if len(str(user_msg).split()) < 3:
        continue
    
    bot_response = get_bot_response_for_user_message(df_sorted, idx)
    questions_with_responses.append({
        'user_message': str(user_msg),
        'bot_response': str(bot_response) if bot_response else None
    })

print(f"User questions with bot responses: {len(questions_with_responses):,}")

# Sample for classification
import random
random.seed(42)
sample_size = min(100, len(questions_with_responses))
sample_data = random.sample(questions_with_responses, sample_size)

print(f"Translating and classifying {sample_size} questions using GPT 5 mini...")

classification_results = []
for item in tqdm(sample_data, desc="Translating and classifying"):
    question = item['user_message']
    bot_response = item['bot_response']
    
    # Translate user question to English
    translated_question = translate_to_english(client, question, model_name="gpt-4.1-mini")
    
    # Translate bot response to English (if exists)
    translated_response = None
    if bot_response:
        translated_response = translate_to_english(client, bot_response, model_name="gpt-4.1-mini")
    
    # Classify the translated question
    classification = classify_query_clarity(client, translated_question, model_name="gpt-4.1-mini")
    
    classification_results.append({
        'original_question': question,
        'translated_question': translated_question,
        'original_bot_response': bot_response,
        'translated_bot_response': translated_response,
        'classification': classification
    })

results_df = pd.DataFrame(classification_results)

In [ ]:
# Summary of classification results
print("\n" + "="*60)
print("QUERY CLASSIFICATION SUMMARY")
print("="*60)

classification_counts = results_df['classification'].value_counts()
for classification, count in classification_counts.items():
    pct = count / len(results_df) * 100
    print(f"{classification}: {count} ({pct:.1f}%)")

# Visualize
plt.figure(figsize=(8, 5))
classification_counts.plot(kind='bar', color=['green', 'red', 'gray'][:len(classification_counts)], edgecolor='black')
plt.title('Query Classification Distribution')
plt.xlabel('Classification')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Display examples of well-specified queries with bot responses
well_specified = results_df[results_df['classification'] == 'WELL_SPECIFIED']

print("\n" + "="*70)
print("EXAMPLES OF WELL-SPECIFIED QUERIES (with bot responses)")
print("="*70)

for idx, (i, row) in enumerate(well_specified.head(10).iterrows()):
    print(f"\n{'─'*70}")
    print(f"Example {idx+1}:")
    print(f"  [USER] {row['translated_question']}")
    if row['translated_bot_response']:
        # Truncate long responses
        response = row['translated_bot_response']
        if len(response) > 400:
            response = response[:400] + "..."
        print(f"  [BOT]  {response}")

In [ ]:
# Display examples of mispecified/vague queries with bot responses
mispecified = results_df[results_df['classification'] == 'MISPECIFIED']

print("\n" + "="*70)
print("EXAMPLES OF MISPECIFIED/VAGUE QUERIES (with bot responses)")
print("="*70)

for idx, (i, row) in enumerate(mispecified.head(10).iterrows()):
    print(f"\n{'─'*70}")
    print(f"Example {idx+1}:")
    print(f"  [USER] {row['translated_question']}")
    if row['translated_bot_response']:
        # Truncate long responses
        response = row['translated_bot_response']
        if len(response) > 400:
            response = response[:400] + "..."
        print(f"  [BOT]  {response}")

In [ ]:
# Analyze patterns in mispecified queries
print("\n" + "="*60)
print("ANALYSIS OF MISPECIFIED QUERIES")
print("="*60)

if len(mispecified) > 0:
    # Word count comparison (using translated questions for fair comparison)
    well_specified_copy = well_specified.copy()
    mispecified_copy = mispecified.copy()
    
    well_specified_copy['word_count'] = well_specified_copy['translated_question'].apply(lambda x: len(str(x).split()))
    mispecified_copy['word_count'] = mispecified_copy['translated_question'].apply(lambda x: len(str(x).split()))
    
    print(f"\nAverage word count (translated):")
    print(f"  Well-specified: {well_specified_copy['word_count'].mean():.1f} words")
    print(f"  Mispecified: {mispecified_copy['word_count'].mean():.1f} words")
    
    # Question mark presence
    well_specified_qmark = well_specified_copy['translated_question'].apply(lambda x: '?' in str(x)).mean() * 100
    mispecified_qmark = mispecified_copy['translated_question'].apply(lambda x: '?' in str(x)).mean() * 100
    
    print(f"\nQueries containing '?':")
    print(f"  Well-specified: {well_specified_qmark:.1f}%")
    print(f"  Mispecified: {mispecified_qmark:.1f}%")
else:
    print("No mispecified queries found in the sample.")

In [ ]:
# Save classification results
results_df.to_csv('query_classification_results.csv', index=False)
print(f"\nClassification results saved to 'query_classification_results.csv'")